# Simple PhysioNet metrics

This notebook contains a simple set of metrics derived from data extracted from the PhysioNet database.

NOTES:
- Add the following columns to data:
  - credentialing datetime
  - 

## Recommended repository

PhysioNet is recommended by:

- Scientific Data: https://www.nature.com/sdata/policies/repositories
- PLOS Biology: https://journals.plos.org/plosbiology/s/recommended-repositories
- NUS: https://libguides.nus.edu.sg/rdm/selected_dr
- NeurIPS: https://neurips.cc/Conferences/2023/CallForDatasetsBenchmarks
- Springer Nature: https://www.springernature.com/gp/authors/research-data-policy/health-sciences-repositories/12327108
- PLOS One: https://journals.plos.org/plosone/s/recommended-repositories#loc-biomedical-sciences
- Elsevier: https://admin1.journals.elsevier.com/

## Datathons

List of datathons:  
https://docs.google.com/document/d/19x5sJFIsbJYAlTC6yegyDio_OhEIRV9Bz0GV57WpQi0/

## Setup

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from tableone import TableOne
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [ ]:
# Set the aesthetic style of the plots
sns.set(style="whitegrid")
fig_axis_fontsize = 12
fig_title_fontsize = 12
fig_bar_color = "dodgerblue"
fig_edgecolor = "black"

In [ ]:
# path to datasets
base_path = os.path.join("..", "data", "physionet")

In [ ]:
# Custom function to parse the datetime
def parse_publish_date(date_str):
    try:
        # Try the format with microseconds first
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S.%f%z")
    except ValueError:
        # If that fails, try the format without microseconds
        return datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S%z")

## Load data

Loading data for analysis, including:

- users
- projects
- historical data (pre-2020)

In [ ]:
# iterate folder
for file in os.listdir(base_path):
    print(file)

In [ ]:
users = pd.read_csv(os.path.join(base_path, "users.csv"), low_memory=False)

# Convert date columns to date type
users['join_date'] = pd.to_datetime(users['join_date'], format="%Y-%m-%d")
# users = users.with_columns(pl.col("join_date").str.strptime(pl.Date, "%Y-%m-%d"))

users.head(3)

In [ ]:
projects = pd.read_csv(os.path.join(base_path, "projects.csv"), low_memory=False)

# Convert date columns to date type
# projects = projects.with_columns(pl.col("publish_date").str.to_datetime("%Y-%m-%d %H:%M:%S%.f%z", strict=False))
# projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')
# projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')
projects['publish_date'] = projects['publish_date'].apply(parse_publish_date).dt.date

# projects[projects.publish_date == pd.NaT]
projects.head()

## Summary statistics

In [ ]:
def safe_len(x):
    """
    calculate number of items in a cell
    """
    if isinstance(x, list):
        return len(x)
    elif isinstance(x, str):  # If it's a string, return the length of the string
        return x.count(',')
    else:
        return 0  # Return 0 for NaNs or other types (like floats)

In [ ]:
projects['number_authors'] = projects['author_ids'].apply(safe_len)

In [ ]:
# Add publication year
projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')
projects['publication_year'] = projects['publish_date'].dt.year

# Add number of authors
projects['number_authors'] = projects['author_ids'].apply(safe_len)

# Consider adding legacy column (i.e. pre september 2019)

# Harmonize DUAs
harmonize = {"Korea Credentialed Health Data Agreement-1-0-0": "Credentialed",
             "Medical AI Foundations Checkpoints: Additional Terms of Service": "Credentialed",
             "PhysioNet Contributor Review Health Data Use Agreement 1.5.0": "Contributor review",
             "PhysioNet Credentialed Health Data Use Agreement 1.5.0": "Credentialed",
             "PhysioNet Restricted Health Data Use Agreement 1.5.0": "Restricted",
             np.nan: "Open"}

# Apply the dictionary to the column using map
projects['access_type'] = projects['data_use_agreement'].map(harmonize)

# Convert Storage size (Mb) to Storage size (Gb)
projects['storage_size_gb'] = projects['storage_size_mb'] / 1000

In [ ]:
# Should we limit to latest versions only?
projects = projects[projects['is_latest_version'] == True]

In [ ]:
# Define the bins (5-year intervals) and labels
bins = list(range(1998, 2027, 7))  # From 2000 to 2025 in steps of 5 years 1998, 2027, 4
# bins = list(range(1998, 2027, 4))  # From 2000 to 2025 in steps of 5 years 1998, 2027, 4
labels = [f'{start}-{end-1}' for start, end in zip(bins[:-1], bins[1:])]
print(labels)

In [ ]:
# Use pd.cut to group the years into buckets
projects['publication_year_group'] = pd.cut(projects['publication_year'],
                                            bins=bins, labels=labels, right=False)

In [ ]:
projects.columns

In [ ]:
columns = ["resource_type", "access_type", "signed_dua_count", "storage_size_gb",
           "number_authors", "publication_year_group"]

groupby = "resource_type"
# groupby = "access_type"
categorical = ["access_type", "resource_type", "publication_year_group"].remove(groupby)

rename= {"resource_type": "Resource_type",
         "access_type": "Access type",
         "signed_dua_count": "Signed DUAs",
         "storage_size_gb": "Storage size (Gb)",
         "number_authors":"Number of authors",
         "publication_year_group":"Publication year"}

order = {"access_type": ["Open", "Restricted", "Credentialed", "Contributor review"],
         "resource_type": ["Database", "Software", "Challenge", "Model"]}

t1 = TableOne(projects, columns=columns, categorical=categorical, groupby=groupby,
              rename=rename, order=order, missing=False)
t1

In [ ]:
print(t1.tabulate(tablefmt="latex"))

In [ ]:
# Define the new projects as a dictionary
new_projects = [
    {'project_id': '0001', 'project_slug': 'mimic2db', 'publish_date': '2011-01-01', 'storage_size_mb': 717000.0},
    {'project_id': '0002', 'project_slug': 'mimic2wdb', 'publish_date': '2015-01-01', 'storage_size_mb': 6800000.0}
]

# Convert the list of dictionaries to a DataFrame
new_projects_df = pd.DataFrame(new_projects)

# Add the new projects to the existing 'projects' DataFrame
projects = pd.concat([projects, new_projects_df], ignore_index=True)

# Display the updated DataFrame
print(projects.tail())

## Metrics

Extract simple metrics from the user and project data.

### Number of new users over time

In [ ]:
# Convert 'join_date' to datetime if it's not already
users['join_date'] = pd.to_datetime(users['join_date'], errors='coerce')

# Extract the year from 'join_date' and create a new column 'year'
users['year'] = users['join_date'].dt.year

# Group by 'year' and count the number of users per year
users_per_year = users.groupby('year').agg(count=('user_id', 'size')).reset_index()

users_per_year.head(5)

In [ ]:
# Create the bar plot using Seaborn
plt.figure(figsize=(8, 6), dpi=300)

# Plotting with Seaborn
ax = sns.barplot(x="year", y="count", data=users_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("New Registered Users", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("New Registered Users Per Year", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks for better readability
plt.xticks(rotation=45, ha='right')

# Ensure y-ticks are integers and do not overlap
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Apply a tight layout to ensure everything fits well
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('new_users_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


In [ ]:
# Calculate the cumulative sum of new users per year
users_per_year['cumulative_count'] = users_per_year['count'].cumsum()

# Create the bar plot using Seaborn
plt.figure(figsize=(8, 6), dpi=300)

# Plotting the cumulative data with Seaborn
ax = sns.barplot(x="year", y="cumulative_count", data=users_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("Cumulative Registered Users", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("Cumulative Registered Users Over Years", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks for better readability
plt.xticks(rotation=45, ha='right')

# Ensure y-ticks are integers and do not overlap
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Apply a tight layout to ensure everything fits well
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('cumulative_users_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

### Number of credentialed users

In [ ]:
# Convert 'join_date' to datetime if it's not already
users['join_date'] = pd.to_datetime(users['join_date'], errors='coerce')

# Filter the DataFrame for credentialed users using .loc[]
credentialed_users = users.loc[users['credentialing_status'] == "Credentialed"].copy()

# Extract the year from 'join_date' and create a new column 'year' using .loc[]
credentialed_users.loc[:, 'year'] = credentialed_users['join_date'].dt.year

# Group by 'year' and count the number of users per year
credentialed_users_per_year = credentialed_users.groupby('year').agg(count=('user_id', 'size')).reset_index()

credentialed_users_per_year.head(3)

In [ ]:
# Create the bar plot using Seaborn
plt.figure(figsize=(8, 6), dpi=300)

# Plotting with Seaborn
ax = sns.barplot(x="year", y="count", data=credentialed_users_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("New Credentialed Users", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("Number of New Credentialed Users Per Year", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks for better readability
plt.xticks(rotation=45, ha='right')

# Ensure y-ticks are integers and do not overlap
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Apply a tight layout to ensure everything fits well
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('new_credentialed_users_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
# Calculate the cumulative sum of new users per year
credentialed_users_per_year['cumulative_count'] = credentialed_users_per_year['count'].cumsum()

# Create the bar plot using Seaborn
plt.figure(figsize=(8, 6), dpi=300)

# Plotting the cumulative data with Seaborn
ax = sns.barplot(x="year", y="cumulative_count", data=credentialed_users_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("Cumulative Credentialed Users", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("Cumulative Credentialed Users Over Years", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks for better readability
plt.xticks(rotation=45, ha='right')

# Ensure y-ticks are integers and do not overlap
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Apply a tight layout to ensure everything fits well
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('cumulative_credentialed_users_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

### Number of published projects over time

In [ ]:
# Convert 'publish_date' to datetime if it's not already
projects['publish_date'] = pd.to_datetime(projects['publish_date'], errors='coerce')

# Extract the year from 'publish_date' and create a new column 'year'
projects['year'] = projects['publish_date'].dt.year

# Group by 'year' and count the number of projects per year
projects_per_year = projects.groupby('year').agg(count=('project_id', 'size'),
                                                 total_storage_mb=('storage_size_mb', 'sum')
                                                 ).reset_index()

projects_per_year.head(26)

In [ ]:
# Create the bar plot using Seaborn
plt.figure(figsize=(8, 4), dpi=300)

# Use Seaborn's barplot
ax = sns.barplot(x="year", y="count", data=projects_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("Publications", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("New Published Projects Per Year", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks
plt.xticks(rotation=45, ha='right')  # This replaces set_xticklabels and correctly rotates the labels

# Ensure y-ticks do not overlap by setting a more appropriate scale
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # Ensures only integer labels are used

# Tight layout for better spacing
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('new_published_projects_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()


In [ ]:
# cumulative projects
projects_per_year['cumulative_count'] = projects_per_year['count'].cumsum()

# Create the bar plot using Seaborn
plt.figure(figsize=(8, 4), dpi=300)

# Use Seaborn's barplot
ax = sns.barplot(x="year", y="cumulative_count", data=projects_per_year, color=fig_bar_color, edgecolor=fig_edgecolor)

# Add labels and title with appropriate sizes
ax.set_xlabel("Year", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_ylabel("Publications", fontsize=fig_axis_fontsize, fontweight='bold')
ax.set_title("Cumulative Published Projects", fontsize=fig_title_fontsize, fontweight='bold')

# Rotate x-ticks
plt.xticks(rotation=45, ha='right')  # This replaces set_xticklabels and correctly rotates the labels

# Ensure y-ticks do not overlap by setting a more appropriate scale
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))  # Ensures only integer labels are used

# Tight layout for better spacing
plt.tight_layout()

# Save the figure as a high-quality image
plt.savefig('cumulative_published_projects_per_year.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

### Number of projects submitted by year since 2020

Need to add submitted projects to the system

### New plot displaying new registered users per year

In [ ]:
# Define the color in HEX for plots
bar_color = '#636efa'  

# Create the bar plot using Plotly Express
fig = px.bar(users_per_year, x="year", y="count", color_discrete_sequence=[bar_color])

# Update layout for the title, labels, and axis formatting
fig.update_layout(
    title={
        'text': 'New Registered Users Per Year',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold title
    },
    xaxis_title={
        'text': 'Year',
        'font': {'size': 18, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold x-axis label
    },
    yaxis_title={
        'text': 'No. registered users (x10³)',  # Update y-axis label to reflect scaling
        'font': {'size': 18, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold y-axis label
    },
    plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
    xaxis=dict(
        showline=True,            # Show line on the x-axis
        linecolor='black',        # Set line color to black
        linewidth=1.5,            # Set line width
        ticks="outside",          # Place ticks outside the axis line
        tickcolor='black',        # Color of the ticks
        ticklen=5,                # Length of the ticks
        mirror=True               # Mirror the line on the top side (adds a border)
    ),
    yaxis=dict(
        showline=True,            # Show line on the y-axis
        linecolor='black',        # Set line color to black
        linewidth=1.5,            # Set line width
        ticks="outside",          # Place ticks outside the axis line
        tickcolor='black',        # Color of the ticks
        ticklen=5,                # Length of the ticks
        mirror=True,              # Mirror the line on the right side (adds a border)
        tickvals=[0, 5000, 10000, 15000, 20000, 25000, 30000],  # Real values in the data
        ticktext=['0', '5', '10', '15', '20', '25', '30'],      # Displayed values in thousands
    )
)

# Show the figure
fig.show()

### Number of credentialed users

In [ ]:
# Convert 'credentialing_decision_date' to datetime if it's not already
users['credentialing_decision_date'] = pd.to_datetime(users['credentialing_decision_date'], errors='coerce')

# Filter the DataFrame for credentialed users using .loc[]
credentialed_users = users.loc[users['credentialing_status'] == "Credentialed"].copy()

# Extract the year from 'credentialing_decision_date' and create a new column 'year' using .loc[]
credentialed_users.loc[:, 'year'] = credentialed_users['credentialing_decision_date'].dt.year

# Group by 'year' and count the number of users per year
credentialed_users_per_year = credentialed_users.groupby('year').agg(count=('user_id', 'size')).reset_index()

credentialed_users_per_year.head(3)

### New plot displaying new credentialed users per year 

In [ ]:
def add_star_to_final_year(fig, x_data, y_data, row, col, incomplete):
    """ Adds a red star marker to the final year if incomplete. """
    if incomplete:
        # Find the final year and its corresponding y-value (height of the bar)
        final_year = x_data.max()
        final_count = y_data[x_data == final_year].values[0]

        # Add the red star
        fig.add_trace(go.Scatter(
            x=[final_year],
            y=[final_count * 1.05],  # Position star slightly above the bar
            mode="markers",
            marker=dict(symbol="star", color="red", size=12),  # Star style
            showlegend=False
        ), row=row, col=col)

In [ ]:
def plot_new_credentialed_users(df, incomplete=True):
    # Define the exact color in HEX for consistency
    exact_bar_color = '#636efa'  # Use the same color for consistency

    # Create the bar plot using Plotly Express
    fig = px.bar(df, x="year", y="count", color_discrete_sequence=[exact_bar_color])

    # Check for incomplete and add the star to the final year
    add_star_to_final_year(fig, df["year"], df["count"], row=1, col=1, incomplete=True)

    # Update layout for the title, labels, and axis formatting
    fig.update_layout(
        title={
            'text': 'New Credentialed Users Per Year',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold title
        },
        xaxis_title={
            'text': 'Year',
            'font': {'size': 18, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold x-axis label
        },
        yaxis_title={
            'text': 'No. credentialed users (x10³)',  # Indicate scaling on the y-axis
            'font': {'size': 18, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold y-axis label
        },
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        xaxis=dict(
            showline=True,            # Show line on the x-axis
            linecolor='black',        # Set line color to black
            linewidth=1.5,            # Set line width
            ticks="outside",          # Place ticks outside the axis line
            tickcolor='black',        # Color of the ticks
            ticklen=5,                # Length of the ticks
            mirror=True               # Mirror the line on the top side (adds a border)
        ),
        yaxis=dict(
            showline=True,            # Show line on the y-axis
            linecolor='black',        # Set line color to black
            linewidth=1.5,            # Set line width
            ticks="outside",          # Place ticks outside the axis line
            tickcolor='black',        # Color of the ticks
            ticklen=5,                # Length of the ticks
            mirror=True,              # Mirror the line on the right side (adds a border)
            tickvals=[0, 2000, 4000, 6000, 8000, 10000, 12000],  # Actual data values
            ticktext=['0', '2', '4', '6', '8', '10', '12'],      # Displayed values in thousands
        )
    )

    # Show the figure
    fig.show()

In [ ]:
plot_new_credentialed_users(credentialed_users_per_year, incomplete=True)

### New plot displaying number of published projects per year

In [ ]:
def plot_new_published_projects(df):
    # Define the exact color in HEX (replace if needed)
    exact_bar_color = '#636efa'  # Use the same color for consistency

    # Create the bar plot using Plotly Express
    fig = px.bar(df, x="year", y="count", color_discrete_sequence=[exact_bar_color])

    add_star_to_final_year(fig, df["year"], df["count"], row=1, col=1, incomplete=True)

    # Update layout for the title and labels, ensuring bold text and proper axis labels
    fig.update_layout(
        title={
            'text': 'New Published Projects Per Year',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold title
        },
        xaxis_title={
            'text': 'Year',
            'font': {'size': 18, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold x-axis label
        },
        yaxis_title={
            'text': 'No. publications',
            'font': {'size': 18, 'color': 'black', 'family': 'Arial', 'weight': 'bold'}  # Bold y-axis label
        },
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        xaxis=dict(
            showline=True,            # Show line on the x-axis
            linecolor='black',        # Set line color to black
            linewidth=1.5,            # Set line width
            ticks="outside",          # Place ticks outside the axis line
            tickcolor='black',        # Color of the ticks
            ticklen=5,                # Length of the ticks
            mirror=True               # Mirror the line on the top side (adds a border)
        ),
        yaxis=dict(
            showline=True,            # Show line on the y-axis
            linecolor='black',        # Set line color to black
            linewidth=1.5,            # Set line width
            ticks="outside",          # Place ticks outside the axis line
            tickcolor='black',        # Color of the ticks
            ticklen=5,                # Length of the ticks
            mirror=True               # Mirror the line on the right side (adds a border)
        )
    )

    # Show the figure
    fig.show()

In [ ]:
plot_new_published_projects(projects_per_year)

## Metrics from Dimensions.ai

### Number of publications referencing PhysioNet

In [ ]:
publications_df = pd.read_csv(os.path.join(base_path, "publications.csv"), low_memory=False)
publications_df.head(20)

In [ ]:
publications_df.columns

### Filter data from 1999 to 2024

In [ ]:
publications_df = publications_df.loc[(publications_df['year'] >= 1999) & (publications_df['year'] <= 2024)]

In [ ]:
# count publications and rename columns
publications = publications_df['year'].value_counts()
publications = publications.to_frame().reset_index().rename(columns= {"index": 'year'})
publications.index.name = 'index'

In [ ]:
publications.value_counts()

### New plot displaying publications referencing PhysioNet per year

In [ ]:
def publications_referencing_physionet(df, incomplete=True):

    # Create the bar plot using Plotly Express
    fig = px.bar(publications, x="year", y="count")

    if incomplete:
        add_star_to_final_year(fig, df["year"], df["count"], row=1, col=1, incomplete=True)

    # Update layout for the title, labels, and formatting
    fig.update_layout(
        title={
            'text': 'Publications Referencing PhysioNet by Year',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20, 'color': 'black', 'family': 'Arial', 'weight': 'bold'},  # Set the title in bold
        },
        xaxis_title={
            'text': 'Year',
            'font': {'size': 18, 'color': 'black', 'family': 'Arial', 'weight': 'bold'},  # Set x-axis label in bold
        },
        yaxis_title={
            'text': 'No. publications (x10³)',
            'font': {'size': 18, 'color': 'black', 'family': 'Arial', 'weight': 'bold'},  # Set y-axis label in bold
        },
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        xaxis=dict(
            showline=True,            # Show line on the x-axis
            linecolor='black',        # Set line color to black
            linewidth=1,              # Set line width
            ticks="outside",          # Place ticks outside the axis line
            tickcolor='black',        # Color of the ticks
            ticklen=5,                # Length of the ticks
            mirror=True               # Mirror the line on the top side
        ),
        yaxis=dict(
            showline=True,            # Show line on the y-axis
            linecolor='black',        # Set line color to black
            linewidth=1,              # Set line width
            ticks="outside",          # Place ticks outside the axis line
            tickcolor='black',        # Color of the ticks
            ticklen=5,                # Length of the ticks
            mirror=True,              # Mirror the line on the right side
            tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000],  # Real values on the scale
            ticktext=['0', '1', '2', '3', '4', '5', '6', '7'],  # Custom text for tick values, scaled down
        )
    )

    # Show the plot
    fig.show()

In [ ]:
publications_referencing_physionet(publications)

### Combined plot with all four subplots arranged side by side

In [ ]:
# Create the subplot figure with 1 row and 4 columns
fig = make_subplots(
    rows=1, cols=4, 
    subplot_titles=(
        "New Registered Users<br>Per Year", 
        "New Credentialed Users<br>Per Year", 
        "New Published Projects<br>Per Year", 
        "Publications Referencing<br>PhysioNet by Year"
    ),
    horizontal_spacing=0.07  # Increase horizontal spacing for better fit
)

# Plot 1: New Registered Users Per Year
fig.add_trace(
    go.Bar(x=users_per_year["year"], y=users_per_year["count"], name='Registered Users', marker_color='#636efa'),
    row=1, col=1
)

# Plot 2: New Credentialed Users Per Year
fig.add_trace(
    go.Bar(x=credentialed_users_per_year["year"], y=credentialed_users_per_year["count"], name='Credentialed Users', marker_color='#636efa'),
    row=1, col=2
)

# Plot 3: New Published Projects Per Year
fig.add_trace(
    go.Bar(x=projects_per_year["year"], y=projects_per_year["count"], name='Published Projects', marker_color='#636efa'),
    row=1, col=3
)

# Plot 4: Publications by Year
fig.add_trace(
    go.Bar(x=publications["year"], y=publications["count"], name='Publications', marker_color='#636efa'),
    row=1, col=4
)

# Update layout for the overall figure
fig.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
    showlegend=False,
    font=dict(family="Arial", size=26, color="black")  # Change font for axis labels
)

# Update subplot titles to size 26 and make them bold
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=26, family='Arial', color='black', weight='bold')  # Set size and bold for subplot titles

# Update the x and y axes properties for all four plots
# Plot 1: Registered Users
fig.update_xaxes(
    title_text="Year",
    title_font=dict(size=26, family='Arial', color='black', weight='bold'),  # Change axis title font size and bold
    showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
    tickfont=dict(size=24),  # Change tick label font size
    row=1, col=1
)
fig.update_yaxes(
    title_text="No. registered users (x10³)",
    title_font=dict(size=26, family='Arial', color='black', weight='bold'),  # Change axis title font size and bold
    showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
    tickvals=[0, 5000, 10000, 15000, 20000, 25000, 30000],
    ticktext=['0', '5', '10', '15', '20', '25', '30'],
    tickfont=dict(size=24),  # Change tick label font size
    row=1, col=1
)

# Plot 2: Credentialed Users
fig.update_xaxes(
    title_text="Year",
    title_font=dict(size=26, family='Arial', color='black', weight='bold'),  # Change axis title font size and bold
    showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
    tickfont=dict(size=24),  # Change tick label font size
    row=1, col=2
)
fig.update_yaxes(
    title_text="No. credentialed users (x10³)",
    title_font=dict(size=26, family='Arial', color='black', weight='bold'),  # Change axis title font size and bold
    showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
    tickvals=[0, 2000, 4000, 6000, 8000, 10000, 12000],
    ticktext=['0', '2', '4', '6', '8', '10', '12'],
    tickfont=dict(size=24),  # Change tick label font size
    row=1, col=2
)

# Plot 3: Published Projects
fig.update_xaxes(
    title_text="Year",
    title_font=dict(size=26, family='Arial', color='black', weight='bold'),  # Change axis title font size and bold
    showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
    tickfont=dict(size=24),  # Change tick label font size
    row=1, col=3
)
fig.update_yaxes(
    title_text="No. publications",
    title_font=dict(size=26, family='Arial', color='black', weight='bold'),  # Change axis title font size and bold
    showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
    tickfont=dict(size=24),  # Change tick label font size
    row=1, col=3
)

# Plot 4: Publications by Year
fig.update_xaxes(
    title_text="Year",
    title_font=dict(size=26, family='Arial', color='black', weight='bold'),  # Change axis title font size and bold
    showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
    tickfont=dict(size=24),  # Change tick label font size
    row=1, col=4
)
fig.update_yaxes(
    title_text="No. publications (x10³)",
    title_font=dict(size=26, family='Arial', color='black', weight='bold'),  # Change axis title font size and bold
    showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
    tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000],
    ticktext=['0', '1', '2', '3', '4', '5', '6', '7'],
    tickfont=dict(size=24),  # Change tick label font size
    row=1, col=4
)

# Adjust margins and size for better spacing and width, with proportional aspect ratio
fig.update_layout(
    margin=dict(l=50, r=50, t=100, b=100),  # Adjust margins for better spacing, reduce top margin
    height=500,  # Adjust the figure height
    width=1800  # Increase the figure width for more horizontal space
)

# Show the combined plot with all four subplots side by side
fig.show()

In [ ]:
def plot_registered_credentialed_users(users_per_year, credentialed_users_per_year, incomplete=True):
    # Create the subplot figure with 1 row and 2 columns
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(
            "New Registered Users<br>Per Year", 
            "New Credentialed Users<br>Per Year"
        ),
        horizontal_spacing=0.09  # Increase horizontal spacing for better fit
    )

    # Plot 1: New Registered Users Per Year
    fig.add_trace(
        go.Bar(x=users_per_year["year"], y=users_per_year["count"], name='Registered Users', marker_color='#636efa'),
        row=1, col=1
    )

    add_star_to_final_year(fig, users_per_year["year"], users_per_year["count"], row=1, col=1, incomplete=True)

    # Plot 2: New Credentialed Users Per Year
    fig.add_trace(
        go.Bar(x=credentialed_users_per_year["year"], y=credentialed_users_per_year["count"], name='Credentialed Users', marker_color='#636efa'),
        row=1, col=2
    )

    add_star_to_final_year(fig, credentialed_users_per_year["year"], credentialed_users_per_year["count"], row=1, col=2, incomplete=True)

    # Update layout for the overall figure
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        showlegend=False,
        font=dict(family="Arial", size=26, color="black")  # Change font for axis labels
    )

    # Update subplot titles to size 26 and make them bold
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=26, family='Arial Black', color='black', weight='bold')  # Set size and bold for subplot titles

    # Update the x and y axes properties for both plots
    # Plot 1: Registered Users
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=28, family='Arial Black', color='black', weight='bold'),  # Change axis title font size and bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=24),  # Change tick label font size
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="No. registered users (x10³)",
        title_font=dict(size=28, family='Arial Black', color='black', weight='bold'),  # Change axis title font size and bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickvals=[0, 5000, 10000, 15000, 20000, 25000, 30000],
        ticktext=['0', '5', '10', '15', '20', '25', '30'],
        tickfont=dict(size=24),  # Change tick label font size
        row=1, col=1
    )

    # Plot 2: Credentialed Users
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=28, family='Arial Black', color='black', weight='bold'),  # Change axis title font size and bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=24),  # Change tick label font size
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="No. credentialed users (x10³)",
        title_font=dict(size=28, family='Arial Black', color='black', weight='bold'),  # Change axis title font size and bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickvals=[0, 2000, 4000, 6000, 8000, 10000, 12000],
        ticktext=['0', '2', '4', '6', '8', '10', '12'],
        tickfont=dict(size=24),  # Change tick label font size
        row=1, col=2
    )

    # Adjust margins and size for square aspect ratio
    fig.update_layout(
        margin=dict(l=50, r=50, t=100, b=100),  # Adjust margins for better spacing
        height=700,  # Adjust the figure height
        width=1400  # Adjust the figure width to be twice the height for square aspect ratio per plot
    )

    # Show the combined plot with both subplots side by side
    fig.show()


In [ ]:
plot_registered_credentialed_users(users_per_year, credentialed_users_per_year, incomplete=True)

In [ ]:
def plot_registered_credentialed_users(users_per_year, credentialed_users_per_year, size_title=34, size_axes_lables=34, size_tick_lables=26, incomplete=True, bold=False):
    font_type = 'Arial'
    if bold:
        font_type = 'Arial Black'
        
    # Create the subplot figure with 1 row and 2 columns
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(
            "New Registered Users", 
            "New Credentialed Users"
        ),
        horizontal_spacing=0.09  # Increase horizontal spacing for better fit
    )

    # Plot 1: New Registered Users Per Year with Cross-Hatching "x"
    fig.add_trace(
        go.Bar(
            x=users_per_year["year"], 
            y=users_per_year["count"], 
            name='Registered Users', 
            marker_color='#636efa',
            marker_pattern_shape="x",  # Cross-hatching pattern
            width=0.5  # Adjust the bar width to make them narrower
        ),
        row=1, col=1
    )

    add_star_to_final_year(fig, users_per_year["year"], users_per_year["count"], row=1, col=1, incomplete=True)

    # Plot 2: New Credentialed Users Per Year with Cross-Hatching "x"
    fig.add_trace(
        go.Bar(
            x=credentialed_users_per_year["year"], 
            y=credentialed_users_per_year["count"], 
            name='Credentialed Users', 
            marker_color='#636efa',
            marker_pattern_shape="x",  # Cross-hatching pattern
            width=0.5  # Adjust the bar width to make them narrower
        ),
        row=1, col=2
    )

    add_star_to_final_year(fig, credentialed_users_per_year["year"], credentialed_users_per_year["count"], row=1, col=2, incomplete=True)

    # Update layout for the overall figure
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        showlegend=False,
        font=dict(family="Arial", size=size_tick_lables, color="black")  # Change font for axis labels
    )

    # Update subplot titles to size 34 and make them bold
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=size_title, family=font_type, color='black', weight='bold')  # Set size and bold for subplot titles

    # Update the x and y axes properties for both plots
    # Plot 1: Registered Users
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black', weight='bold'),  # Change axis title font size and bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Change tick label font size
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Registered users (x10³)",
        title_font=dict(size=size_axes_lables, family=font_type, color='black', weight='bold'),  # Change axis title font size and bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickvals=[0, 5000, 10000, 15000, 20000, 25000, 30000],
        ticktext=['0', '5', '10', '15', '20', '25', '30'],
        tickfont=dict(size=size_tick_lables),  # Change tick label font size
        title_standoff=0,  # Set standoff for the first plot
        row=1, col=1
    )

    # Plot 2: Credentialed Users
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black', weight='bold'),  # Change axis title font size and bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Change tick label font size
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="Credentialed users (x10³)",
        title_font=dict(size=size_axes_lables, family=font_type, color='black', weight='bold'),  # Change axis title font size and bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickvals=[0, 2000, 4000, 6000, 8000, 10000, 12000],
        ticktext=['0', '2', '4', '6', '8', '10', '12'],
        tickfont=dict(size=size_tick_lables),  # Change tick label font size
        title_standoff=0,  # Adjust title_standoff to move y-axis title closer to tick labels
        row=1, col=2
    )

    # Adjust margins and size for square aspect ratio
    fig.update_layout(
        margin=dict(l=100, r=30, t=120, b=150),  # Adjust margins for better spacing
        height=700,  # Adjust the figure height
        width=1400  # Adjust the figure width to be twice the height for square aspect ratio per plot
    )

    # Show the combined plot with both subplots side by side
    fig.show()


In [ ]:
plot_registered_credentialed_users(users_per_year, credentialed_users_per_year, incomplete=True, bold=False)

In [ ]:
plot_registered_credentialed_users(users_per_year, credentialed_users_per_year, incomplete=True, bold=True)

In [ ]:
def plot_projects_publications(projects_per_year, publications, incomplete=True):
    # Create the subplot figure with 1 row and 2 columns
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(
            "New Published Projects<br>Per Year", 
            "Publications Referencing<br>PhysioNet by Year"
        ),
        horizontal_spacing=0.07  # Increase horizontal spacing for better fit
    )

    # Calculate the y-axis max value for padding
    padding_factor = 1.15  # This factor adds 15% extra space to the y-axis for a larger buffer
    
    # Plot 1: New Published Projects Per Year
    max_y_projects = projects_per_year["count"].max() * padding_factor
    fig.add_trace(
        go.Bar(x=projects_per_year["year"], y=projects_per_year["count"], name='Published Projects', marker_color='#636efa'),
        row=1, col=1
    )
    add_star_to_final_year(fig, projects_per_year["year"], projects_per_year["count"], row=1, col=1, incomplete=True)

    # Plot 2: Publications by Year
    max_y_publications = publications["count"].max() * padding_factor
    fig.add_trace(
        go.Bar(x=publications["year"], y=publications["count"], name='Publications', marker_color='#636efa'),
        row=1, col=2
    )
    add_star_to_final_year(fig, publications["year"], publications["count"], row=1, col=2, incomplete=True)

    # Update layout for the overall figure
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        showlegend=False
    )

    # Update subplot titles to size 26 and make them bold
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=26, family='Arial Black', color='black')  # Set size and bold for subplot titles

    # Update the x and y axes properties for both plots
    max_year = 2024  # Assuming the max year is 2024

    # Plot 1: Published Projects
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=26, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[projects_per_year["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="No. publications",
        title_font=dict(size=26, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[0, max_y_projects],  # Set y-axis range with padding
        row=1, col=1
    )

    # Plot 2: Publications by Year
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=26, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[publications["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="No. publications (x10³)",
        title_font=dict(size=26, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000],
        ticktext=['0', '1', '2', '3', '4', '5', '6', '7'],
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[0, max_y_publications],  # Set y-axis range with padding
        row=1, col=2
    )

    # Adjust margins and size for square aspect ratio
    fig.update_layout(
        margin=dict(l=50, r=50, t=100, b=150),  # Adjust margins for better spacing
        height=700,  # Adjust the figure height
        width=1400  # Adjust the figure width to be twice the height for square aspect ratio per plot
    )

    # Show the combined plot with the two subplots side by side
    fig.show()


In [ ]:
plot_projects_publications(projects_per_year, publications, True)

### Adding patent data from Dimensions.ai

In [ ]:
patents = pd.read_csv(os.path.join(base_path, "patents.csv"), low_memory=False)
patents.head()

In [ ]:
# Create a full range of years from 1999 to 2024
full_years = pd.DataFrame({'granted_year': range(1999, 2025)})

# Count journals and rename columns
granted_patents = patents['granted_year'].value_counts().reset_index()
granted_patents.columns = ['granted_year', 'counts']

# Merge full_years with granted_patents to include missing years
granted_patents = pd.merge(full_years, granted_patents, on='granted_year', how='left')

# Replace NaN values with 0 for missing years
granted_patents['counts'] = granted_patents['counts'].fillna(0)

In [ ]:
def plot_projects_publications_patents(projects_per_year, publications, granted_patents, incomplete=True):
    # Create the subplot figure with 1 row and 3 columns
    fig = make_subplots(
        rows=1, cols=3, 
        subplot_titles=(
            "New Published Projects", 
            "Publications Referencing",
            "Patents Granted"
        ),
        horizontal_spacing=0.07  # Increase horizontal spacing for better fit
    )

    # Calculate the y-axis max value for padding
    padding_factor = 1.15  # This factor adds 15% extra space to the y-axis for a larger buffer
    
    # Plot 1: New Published Projects Per Year with Cross-Hatching
    max_y_projects = projects_per_year["count"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=projects_per_year["year"], 
            y=projects_per_year["count"], 
            name='Published Projects', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=1
    )
    add_star_to_final_year(fig, projects_per_year["year"], projects_per_year["count"], row=1, col=1, incomplete=True)

    # Plot 2: Publications by Year with Cross-Hatching
    max_y_publications = publications["count"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=publications["year"], 
            y=publications["count"], 
            name='Publications', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=2
    )
    add_star_to_final_year(fig, publications["year"], publications["count"], row=1, col=2, incomplete=True)

    # Plot 3: Patents Granted by Year with Cross-Hatching
    max_y_patents = granted_patents["counts"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=granted_patents["granted_year"], 
            y=granted_patents["counts"], 
            name='Patents Granted', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=3
    )
    add_star_to_final_year(fig, granted_patents["granted_year"], granted_patents["counts"], row=1, col=3, incomplete=True)

    # Update layout for the overall figure
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        showlegend=False
    )

    # Update subplot titles to size 26 and make them bold
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=36, family='Arial Black', color='black')  # Set size and bold for subplot titles

    # Update the x and y axes properties for all plots

    max_year = 2024  # Assuming the max year is 2024

    # Plot 1: Published Projects
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=36, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[projects_per_year["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Published projects",
        title_font=dict(size=36, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[0, max_y_projects],  # Set y-axis range with padding
        row=1, col=1
    )

    # Plot 2: Publications by Year
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=36, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[publications["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="Publications (x10³)",
        title_font=dict(size=36, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000],
        ticktext=['0', '1', '2', '3', '4', '5', '6', '7'],
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[0, max_y_publications],  # Set y-axis range with padding
        row=1, col=2
    )

    # Plot 3: Patents Granted by Year
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=36, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[granted_patents["granted_year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=3
    )
    fig.update_yaxes(
        title_text="Patents granted",
        title_font=dict(size=36, family='Arial Black', color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickfont=dict(size=24),  # Keep tick label font size, not bold
        range=[0, max_y_patents],  # Set y-axis range with padding
        row=1, col=3
    )

    # Adjust margins and size for square aspect ratio
    fig.update_layout(
        margin=dict(l=50, r=50, t=100, b=150),  # Adjust margins for better spacing, reduce top margin
        height=700,  # Adjust the figure height
        width=2100  # Adjust the figure width to be three times the height for square aspect ratio per plot
    )

    # Show the combined plot with the three subplots side by side
    fig.show()


In [ ]:
plot_projects_publications_patents(projects_per_year, publications, granted_patents, incomplete=True)

In [ ]:
def plot_projects_publications_patents(projects_per_year, publications, granted_patents, size_title=34, size_axes_lables=34, size_tick_lables=26, incomplete=True, bold=False):
    font_type = 'Arial'
    if bold:
        font_type = 'Arial Black'

    # Create the subplot figure with 1 row and 3 columns
    fig = make_subplots(
        rows=1, cols=3, 
        subplot_titles=(
            "New PhysioNet Resources", 
            "Papers Referencing<br> PhysioNet",
            "Patents Granted"
        ),
        horizontal_spacing=0.07  # Increase horizontal spacing for better fit
    )

    # Calculate the y-axis max value for padding
    padding_factor = 1.15  # This factor adds 15% extra space to the y-axis for a larger buffer
    
    # Plot 1: New Published Projects Per Year with Cross-Hatching
    max_y_projects = projects_per_year["count"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=projects_per_year["year"], 
            y=projects_per_year["count"], 
            name='Published Projects', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=1
    )
    add_star_to_final_year(fig, projects_per_year["year"], projects_per_year["count"], row=1, col=1, incomplete=True)

    # Plot 2: Publications by Year with Cross-Hatching
    max_y_publications = publications["count"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=publications["year"], 
            y=publications["count"], 
            name='Publications', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=2
    )
    add_star_to_final_year(fig, publications["year"], publications["count"], row=1, col=2, incomplete=True)

    # Plot 3: Patents Granted by Year with Cross-Hatching
    max_y_patents = granted_patents["counts"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=granted_patents["granted_year"], 
            y=granted_patents["counts"], 
            name='Patents Granted', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=3
    )
    add_star_to_final_year(fig, granted_patents["granted_year"], granted_patents["counts"], row=1, col=3, incomplete=True)

    # Update layout for the overall figure
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        showlegend=False
    )

    # Update subplot titles to size 26 and make them bold
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=size_title, family=font_type, color='black')  # Set size and bold for subplot titles

    # Update the x and y axes properties for all plots

    max_year = 2024  # Assuming the max year is 2024

    # Plot 1: Published Projects
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[projects_per_year["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Published Resources",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[0, max_y_projects],  # Set y-axis range with padding
        title_standoff=0,  # Adjust title_standoff to move y-axis title closer to tick labels
        row=1, col=1
    )

    # Plot 2: Publications by Year
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[publications["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="Papers (x10³)",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000],
        ticktext=['0', '1', '2', '3', '4', '5', '6', '7'],
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[0, max_y_publications],  # Set y-axis range with padding
        title_standoff=0,  # Adjust title_standoff to move y-axis title closer to tick labels
        row=1, col=2
    )

    # Plot 3: Patents Granted by Year
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[granted_patents["granted_year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=3
    )
    fig.update_yaxes(
        title_text="Patents granted",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[0, max_y_patents],  # Set y-axis range with padding
        title_standoff=0,  # Adjust title_standoff to move y-axis title closer to tick labels
        row=1, col=3
    )

    # Adjust margins and size for square aspect ratio
    fig.update_layout(
        margin=dict(l=100, r=30, t=120, b=150), # Adjust margins for better spacing, reduce top margin
        height=700,  # Adjust the figure height
        width=2100  # Adjust the figure width to be three times the height for square aspect ratio per plot
    )

    # Show the combined plot with the three subplots side by side
    fig.show()


In [ ]:
plot_projects_publications_patents(projects_per_year, publications, granted_patents, incomplete=True, bold=False)

In [ ]:
plot_projects_publications_patents(projects_per_year, publications, granted_patents, incomplete=True, bold=True)

In [ ]:
def plot_publications_patents(publications, granted_patents, size_title=34, size_axes_lables=34, size_tick_lables=26, incomplete=True, bold=False):
    font_type = 'Arial'
    if bold:
        font_type = 'Arial Black'

    # Create the subplot figure with 1 row and 3 columns
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(
            "Articles Referencing PhysioNet",
            "Patents Granted"
        ),
        horizontal_spacing=0.1  # Increase horizontal spacing for better fit
    )

    # Calculate the y-axis max value for padding
    padding_factor = 1.15  # This factor adds 15% extra space to the y-axis for a larger buffer
    
    # Plot 1: Publications by Year with Cross-Hatching
    max_y_publications = publications["count"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=publications["year"], 
            y=publications["count"], 
            name='Publications', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=1
    )
    add_star_to_final_year(fig, publications["year"], publications["count"], row=1, col=1, incomplete=True)

    # Plot 2: Patents Granted by Year with Cross-Hatching
    max_y_patents = granted_patents["counts"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=granted_patents["granted_year"], 
            y=granted_patents["counts"], 
            name='Patents Granted', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=2
    )
    add_star_to_final_year(fig, granted_patents["granted_year"], granted_patents["counts"], row=1, col=2, incomplete=True)

    # Update layout for the overall figure
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        showlegend=False
    )

    # Update subplot titles to size 26 and make them bold
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=size_title, family=font_type, color='black')  # Set size and bold for subplot titles

    # Update the x and y axes properties for all plots

    max_year = 2024  # Assuming the max year is 2024

    # Plot 1: Publications by Year
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[publications["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Articles (x10³)",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickvals=[0, 1000, 2000, 3000, 4000, 5000, 6000, 7000],
        ticktext=['0', '1', '2', '3', '4', '5', '6', '7'],
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[0, max_y_publications],  # Set y-axis range with padding
        title_standoff=0,  # Adjust title_standoff to move y-axis title closer to tick labels
        row=1, col=1
    )

    # Plot 2: Patents Granted by Year
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[granted_patents["granted_year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="Patents",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[0, max_y_patents],  # Set y-axis range with padding
        title_standoff=0,  # Adjust title_standoff to move y-axis title closer to tick labels
        row=1, col=2
    )

    # Adjust margins and size for square aspect ratio
    fig.update_layout(
        margin=dict(l=100, r=30, t=100, b=100), # Adjust margins for better spacing, reduce top margin
        height=700,  # Adjust the figure height
        width=1400  # Adjust the figure width to be three times the height for square aspect ratio per plot
    )

    # Show the combined plot with the three subplots side by side
    fig.show()

In [ ]:
plot_publications_patents(publications, granted_patents, incomplete=True, bold=False)

In [ ]:
def plot_projects_volume(projects_per_year, size_title=34, size_axes_lables=34, size_tick_lables=26, incomplete=True, bold=False):
    font_type = 'Arial'
    if bold:
        font_type = 'Arial Black'

    # Create the subplot figure with 1 row and 3 columns
    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=(
            "New Published Projects", 
            "PhysioNet Projects by Volume"
        ),
        horizontal_spacing=0.12  # Increase horizontal spacing for better fit
    )

    # Calculate the y-axis max value for padding
    padding_factor = 1.15  # This factor adds 15% extra space to the y-axis for a larger buffer
    
    # Plot 1: New Published Projects Per Year with Cross-Hatching
    max_y_projects = projects_per_year["count"].max() * padding_factor
    fig.add_trace(
        go.Bar(
            x=projects_per_year["year"], 
            y=projects_per_year["count"], 
            name='Published Projects', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=1
    )
    add_star_to_final_year(fig, projects_per_year["year"], projects_per_year["count"], row=1, col=1, incomplete=True)

    # Plot 2: New Published Projects Per Year with Cross-Hatching (Volume)
    max_y_volume = projects_per_year["total_storage_mb"].max() * padding_factor / 1000  # Scale volume to thousands
    fig.add_trace(
        go.Bar(
            x=projects_per_year["year"], 
            y=projects_per_year["total_storage_mb"] / 1000,  # Divide by 1000 for x10³ display
            name='Volume', 
            marker_color='#636efa',
            marker_pattern_shape="x"  # Apply cross-hatching pattern
        ),
        row=1, col=2
    )
    add_star_to_final_year(fig, projects_per_year["year"], projects_per_year["total_storage_mb"] / 1000, row=1, col=2, incomplete=True)

    # Update layout for the overall figure
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        showlegend=False
    )

    # Update subplot titles to size 26 and make them bold
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=size_title, family=font_type, color='black')  # Set size and bold for subplot titles

    # Update the x and y axes properties for all plots
    max_year = 2024  # Assuming the max year is 2024

    # Plot 1: Published Projects
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[projects_per_year["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=1
    )
    fig.update_yaxes(
        title_text="Projects",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[0, max_y_projects],  # Set y-axis range with padding
        title_standoff=0,  # Adjust title_standoff to move y-axis title closer to tick labels
        row=1, col=1
    )

    # Plot 2: Volume by Year (Scaled in thousands)
    fig.update_xaxes(
        title_text="Year",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on top
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[projects_per_year["year"].min() - 0.5, max_year + 0.5],  # Set x-axis range with padding
        row=1, col=2
    )
    fig.update_yaxes(
        title_text="Volume in TB",
        title_font=dict(size=size_axes_lables, family=font_type, color='black'),  # Change axis title font to bold
        showline=True, linecolor='black', linewidth=1, ticks="outside", tickcolor='black', ticklen=5, mirror=True,  # Add black line on right
        tickfont=dict(size=size_tick_lables),  # Keep tick label font size, not bold
        range=[0, max_y_volume],  # Set y-axis range with padding
        tickvals=[0, 2000, 4000, 6000, 8000, 10000],  # Adjust the tick values to reflect thousands
        ticktext=['0', '2', '4', '6', '8', '10'],  # Display x10³ values
        title_standoff=0,  # Adjust title_standoff to move y-axis title closer to tick labels
        row=1, col=2
    )

    # Adjust margins and size for square aspect ratio
    fig.update_layout(
        margin=dict(l=100, r=30, t=100, b=100),  # Adjust margins for better spacing
        height=700,  # Adjust the figure height
        width=1400  # Adjust the figure width to be twice the height for square aspect ratio per plot
    )

    # Show the combined plot with the two subplots side by side
    fig.show()


In [ ]:
projects_per_year.head(20)

In [ ]:
plot_projects_volume(projects_per_year, incomplete=True, bold=False)

### OTHERS

- Proportion of projects accepted
- Breakdown by project type?
- Proportion of conference/journal papers that use PhysioNet data/software
- Volume of data that we are hosting
- Number of users signing DUA
- Average daily visits
- Average monthly users (visits)
- New Google Scholar articles citing
- Number of courses
- Number of patents
- Indication that what we are doing is useful
- Stanford paper on shaky foundations (PhysioNet supports a whole field of research)
- Number of datathons/workshops [Chrystinne]
- International reach (users and contributors)
- Who is contributing? What proportion are NIH funded. Providing a mechanism for NIH funded projects to share
- Number of journals referencing PhysioNet as recommended repository
- Analysis of emails. What is the topic? Etc.
